# 03b 相关性分析（论文风格：全矩阵 + 注释数值）

- 使用表格给定的 **BCF**（不引入 bcf_calc）
- 变量顺序：BCF/Cd → pH/SOM/CEC → 质地/水分 → 养分 → 地形/人为
- 输出目录：`outputs/corr_advanced/`


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
from src.utils import load_config, ensure_dir

cfg = load_config('../config/config.yaml')
OUT = ensure_dir('../outputs')
OUT
# 读取清洗后的数据（优先 bcf_final.parquet；没有就用 clean_data.parquet）

p1 = OUT/'clean_data.parquet'
if p1.exists():
    df = pd.read_parquet(p1)
else:
    raise FileNotFoundError('未找到 clean_data.parquet，请先跑 01（以及可选 02/后续步骤）')
df.shape
# 计算并输出相关性图（PNG）
from src.corr_advanced import build_group_correlations, export_corr_pdf

corr_maps = build_group_correlations(df, cfg, OUT, group_keys=('crop','ph_bin'))
len(corr_maps)
# 汇总成一个 PDF（可选）
pdf_path = export_corr_pdf(OUT, cfg)
pdf_path
# 快速检查输出是否生成
corr_dir = OUT/'corr_advanced'
pngs = sorted(corr_dir.glob('*.png'))
len(pngs), pngs[:5]



In [1]:
# =========================
# 03 Corr Advanced (FINAL)
# - Config-driven group keys
# - Exports PNGs + PDF
# =========================

import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
from src.utils import load_config, ensure_dir

# 1) load config & paths
cfg = load_config('../config/config.yaml')
OUT = ensure_dir('../outputs')

# 2) load cleaned data
p1 = OUT / 'clean_data.parquet'
if not p1.exists():
    raise FileNotFoundError('未找到 clean_data.parquet，请先跑 01')

df = pd.read_parquet(p1)
print("Loaded:", df.shape)
print("Columns (first 20):", list(df.columns)[:20])

# 3) resolve group keys from yaml
#    corr_advanced:
#      group_keys: ["crop","ph_bin"]  # or ["crop"] or []
corr_cfg = (cfg.get("corr_advanced", {}) or {})
gk = corr_cfg.get("group_keys", None)

if gk is None:
    group_keys = ("crop", "ph_bin")  # default (same as src default)
elif isinstance(gk, (list, tuple)):
    group_keys = tuple([str(x).strip() for x in gk if str(x).strip() != ""])
else:
    raise ValueError("config.yaml: corr_advanced.group_keys 必须是 list，例如 ['crop','ph_bin'] 或 []")

# show actual usable group keys in df
usable = [k for k in group_keys if k in df.columns]
print("Config group_keys:", group_keys)
print("Usable group_keys:", usable if usable else "ALL (no grouping)")


# 5) diagnostic: group sizes (helps explain why some groups are skipped by min_rows)
min_rows = int(corr_cfg.get("min_rows", 30))
if usable:
    gsize = df.groupby(usable, dropna=False).size().sort_values(ascending=False)
    print("\nTop group sizes:")
    display(gsize.head(15))
    print("\nSmall groups (< min_rows=%d) count:" % min_rows, int((gsize < min_rows).sum()))
else:
    print("\nNo grouping -> ALL samples used:", len(df))

# 6) run correlation export
from src.corr_advanced import build_group_correlations, export_corr_pdf

corr_maps = build_group_correlations(df, cfg, OUT, group_keys=group_keys)
print("\nGenerated correlation matrices:", len(corr_maps))

pdf_path = export_corr_pdf(OUT, cfg)
print("PDF:", pdf_path)

# 7) quick check outputs
corr_dir = OUT / str(corr_cfg.get("out_subdir", "corr_advanced"))
pngs = sorted(corr_dir.glob("*.png"))
print("PNGs:", len(pngs))
print("First 5 PNGs:", pngs[:5])


Loaded: (4663, 22)
Columns (first 20): ['X', 'Y', 'pH', 'BCF', 'Mn', 'N', 'P', 'K', 'SOM', 'CEC', 'water content', 'Particle content', 'clay content', 'sand content', 'soil type', 'bulk density', 'Vegetation cover type', 'Distance from pollution source', 'source_sheet', 'crop']
Config group_keys: ('crop', 'ph_bin')
Usable group_keys: ['crop', 'ph_bin']

Top group sizes:


crop    ph_bin     
corn    strong_acid    1276
potato  strong_acid    1034
corn    acid            937
potato  acid            477
corn    neutral         349
        alkaline        331
potato  neutral         158
        alkaline        101
dtype: int64


Small groups (< min_rows=30) count: 0

Generated correlation matrices: 8
PDF: ..\outputs\corr_advanced\corr_advanced_all.pdf
PNGs: 8
First 5 PNGs: [WindowsPath('../outputs/corr_advanced/corr_full_corn_acid.png'), WindowsPath('../outputs/corr_advanced/corr_full_corn_alkaline.png'), WindowsPath('../outputs/corr_advanced/corr_full_corn_neutral.png'), WindowsPath('../outputs/corr_advanced/corr_full_corn_strong_acid.png'), WindowsPath('../outputs/corr_advanced/corr_full_potato_acid.png')]
